# Query a Lakehouse file with SQL
**Use DuckDB inside a Pure Python notebook to explore Parquet without a Spark session.**
<br>Attach `lh_pond`, run `0. Setup` with `DATA_SCOPE = "one_year"`, then run sections 0-6 in order. Optional examples follow the resource check.

## 0. Connect the notebook to its tools

Import DuckDB, then confirm the workspace and attached Lakehouse.
<br>In the next section, we’ll set the file path and open a DuckDB connection.
<br>Then we’ll query the files with SQL.

### Import duckdb and notebookutils
`duckdb` provides the SQL engine. `notebookutils` provides access to the Fabric context.

In [ ]:
import duckdb         # Bring DuckDB into scope to connect and run SQL.
import notebookutils  # Access Fabric utilities, including workspace and Lakehouse context.

### Confirm the Fabric context
These values come from the notebook's attached Lakehouse. They are useful for orientation only;
DuckDB will query the mounted files in the next steps.

In [ ]:
# Get the Fabric context for this notebook session.
ctx = notebookutils.runtime.context

# Show the current workspace and the attached default Lakehouse.
print("Workspace:", ctx["currentWorkspaceName"])
print("Lakehouse:", ctx["defaultLakehouseName"])

## 1. Point SQL at the Lakehouse files
`FLIGHT_FILES` holds the path to our Parquet files in the Lakehouse **Files** area.
<br>The `flight_list_*.parquet` pattern selects the monthly files in the one-year folder.
<br>DuckDB’s `read_parquet()` lets SQL query those files directly.
<br>Defining the path does not read or copy the data.

In [ ]:
# One Year
FLIGHT_FILES = "/lakehouse/default/Files/flights_eu/one_year/flight_list_*.parquet"

# Optional full dataset: 2022-2025
#FLIGHT_FILES = "/lakehouse/default/Files/flights_eu/full/flight_list_*.parquet"


print("Flight files:", FLIGHT_FILES)

### Open DuckDB
DuckDB is already available in the Pure Python runtime and runs inside this Python
process. This connection uses an in-memory database, while the source files remain
in the Lakehouse.
<br>We can query those files without first loading the full dataset into a pandas DataFrame.

In [ ]:
# Open an in-memory DuckDB database inside this Python process.
con = duckdb.connect()

# Show the DuckDB version available in this session.
print("DuckDB:", duckdb.__version__)

## 2. Ask the first SQL question
**How many flight rows are in these files?**
<br>`read_parquet()` reads the matching files as one table for this query.
<br>`union_by_name = true` aligns columns by name across the files.
<br>The `f` before the SQL string inserts the `FLIGHT_FILES` path into the query.
<br>DuckDB counts the rows; `.show()` displays the result.

In [ ]:
con.sql(f"""
    SELECT count(*) AS flights
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
""").show()

**About 16.2 million for one year**
The optional `full` setup covers 2022-2025. Run a count to verify the downloaded data.

## 3. Inspect the column names and types
**Use `DESCRIBE` to see which fields the files expose.** It reports schema, not data quality.
<br>`dof` is stored as text in these files; later examples convert it for date calculations.

In [ ]:
con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
""").show()

## 4. Trust, but verify
Real flight data can have missing values. Check coverage before filtering rows or trusting
a count.
<br>`count(*)` counts every row. `count(column)` counts rows where that column is not NULL.
<br>Each percentage shows how often a field is populated. Empty strings still count as populated.

In [ ]:
con.sql(f"""
    SELECT count(*) AS rows_total,
           round(100.0 * count(adep) / count(*), 1) AS origin_present_pct,
           round(100.0 * count(ades) / count(*), 1) AS destination_present_pct,
           round(100.0 * count(typecode) / count(*), 1) AS typecode_present_pct,
           round(100.0 * count(registration) / count(*), 1) AS registration_present_pct
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
""").show()

## 5. Name Barcelona's busiest routes with a CSV join
**Join flight Parquet to the airport CSV and show the top ten routes involving Barcelona (`LEBL`).** The CSV supplies names for both ends of each route.
<br>Left joins keep flight rows even when an airport is missing from the reference data.

In [ ]:
AIRPORTS = "/lakehouse/default/Files/flights_eu/reference/airports.csv"
FLIGHT_FILES = "/lakehouse/default/Files/flights_eu/one_year/flight_list_*.parquet"

con.sql(f"""
    SELECT f.adep,
           coalesce(dep.municipality, dep.name) AS origin,
           f.ades,
           coalesce(arr.municipality, arr.name) AS destination,
           count(*) AS flights
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true) AS f
    LEFT JOIN read_csv_auto('{AIRPORTS}') AS dep
      ON f.adep = dep.ident
    LEFT JOIN read_csv_auto('{AIRPORTS}') AS arr
      ON f.ades = arr.ident
    WHERE (f.adep = 'LEBL' OR f.ades = 'LEBL')
      AND f.adep IS NOT NULL
      AND f.ades IS NOT NULL
    GROUP BY f.adep, coalesce(dep.municipality, dep.name),
             f.ades, coalesce(arr.municipality, arr.name)
    ORDER BY flights DESC
    LIMIT 10
""").show()

## 6. Bring only the top 25 airport results into pandas
**DuckDB joins and aggregates the files; `.df()` returns the small result to Python.** The full flight dataset is not loaded into a pandas DataFrame.
<br>The inner join includes only airports matched in the reference CSV; grouping by airport ID keeps identical labels separate.

In [ ]:
airport_departures = con.sql(f"""
    SELECT a.municipality,
           a.name,
           count(*) AS departures
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true) AS f
    JOIN read_csv_auto('{AIRPORTS}') AS a
      ON f.adep = a.ident
    WHERE f.adep IS NOT NULL
    GROUP BY a.ident, a.municipality, a.name
    ORDER BY departures DESC
    LIMIT 25
""").df()

display(airport_departures)

## Pause: inspect resource usage
Open the notebook’s **Resource usage** panel.
<br>Check the node size, CPU activity, and memory use during these queries.
<br>Use these measurements to help decide whether the workload still fits on this node.

## Optional: publish the airport result as Delta
**Set `RUN_DELTA_WRITE = True` to save section 6's result as `dbo.busiest_airports`.** Both Delta cells are skipped by default.
<br>This requires a schema-enabled Lakehouse and overwrites that derived table through OneLake's `abfss://` path.

In [ ]:
RUN_DELTA_WRITE = False  # Set to True to publish the result.

if RUN_DELTA_WRITE:
    from deltalake import write_deltalake

    workspace_id = ctx["defaultLakehouseWorkspaceId"]
    lakehouse_id = ctx["defaultLakehouseId"]
    assert workspace_id and lakehouse_id, "Attach a default Lakehouse first."

    table_schema = "dbo"
    table_name = "busiest_airports"
    table_path = (
        f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/"
        f"{lakehouse_id}/Tables/{table_schema}/{table_name}"
    )

    storage_options = {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }

    write_deltalake(
        table_path,
        airport_departures,
        mode="overwrite",
        storage_options=storage_options,
    )
    print(f"Wrote Delta table: {table_schema}.{table_name}")
else:
    print("Delta publishing skipped.")

### Read back the published Delta table
**Show ten rows from the table just written.** This cell runs only when `RUN_DELTA_WRITE` is True; it verifies read-back rather than displaying the original DataFrame.

In [ ]:
if RUN_DELTA_WRITE:
    from deltalake import DeltaTable

    delta_table = DeltaTable(table_path, storage_options=storage_options)
    display(delta_table.to_pandas().head(10))

## 7. Keep exploring
*Use these additional exercises to explore the dataset.*
<br>Small, iterative questions often fit on one Python node.
<br>Explore the data here, then choose a distributed engine if the workload grows beyond that node.

### What period does this dataset cover?
Convert `dof` to a date, then inspect the earliest date, latest date, and number of
distinct months.
<br>`try_cast` returns NULL for values it cannot parse. These results summarize valid
dates; twelve months does not prove every day is present.

In [ ]:
con.sql(f"""
    WITH flights AS (
        SELECT try_cast(dof AS DATE) AS flight_date
        FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    )
    SELECT
        min(flight_date) AS first_day,
        max(flight_date) AS last_day,
        count(DISTINCT date_trunc('month', flight_date)) AS months
    FROM flights
""").show()

### Which aircraft types dominate?
Knowing a column is populated is only the start. Count the most common aircraft types and see whether a few account for most flights.
The percentages are based on flights with a known `typecode`, so the top 10 may not add to 100%.

In [ ]:
con.sql(f"""
    SELECT typecode,
           count(*) AS flights,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    WHERE typecode IS NOT NULL
    GROUP BY typecode
    ORDER BY flights DESC
    LIMIT 10
""").show()

### Does flight volume change by month?
DuckDB parses the dates and counts flights by month.
<br>We bring those monthly totals into a pandas DataFrame and plot them to spot gaps or spikes worth investigating.
Months with no valid rows are absent from this result; the query does not fill them with zeros.

In [ ]:
monthly_flights = con.sql(f"""
    WITH flights AS (
        SELECT try_cast(dof AS DATE) AS flight_date
        FROM read_parquet('{FLIGHT_FILES}', union_by_name = true)
    )
    SELECT date_trunc('month', flight_date) AS month,
           count(*) AS flights
    FROM flights
    WHERE flight_date IS NOT NULL
    GROUP BY date_trunc('month', flight_date)
    ORDER BY month
""").df()

monthly_flights.plot(
    x="month",
    y="flights",
    figsize=(10, 4),
    title="European flights by month",
)

### Optional: map the question
DuckDB prepares the map data with SQL; Plotly draws the result.
Set `RUN_AIRPORT_MAP = True` to show up to 200 departure airports with coordinates.
Marker size represents departures, and Barcelona is highlighted in coral.

In [ ]:
RUN_AIRPORT_MAP = True

if RUN_AIRPORT_MAP:
    import plotly.express as px

    airport_map = con.sql(f"""
        SELECT a.ident,
               a.name,
               a.municipality,
               try_cast(a.latitude_deg AS DOUBLE) AS latitude,
               try_cast(a.longitude_deg AS DOUBLE) AS longitude,
               count(*) AS departures,
               CASE
                   WHEN a.ident = 'LEBL' THEN 'Barcelona'
                   ELSE 'Other airport'
               END AS airport_type
        FROM read_parquet('{FLIGHT_FILES}', union_by_name = true) AS f
        JOIN read_csv_auto('{AIRPORTS}') AS a
          ON f.adep = a.ident
        WHERE a.latitude_deg IS NOT NULL
          AND a.longitude_deg IS NOT NULL
        GROUP BY a.ident, a.name, a.municipality, a.latitude_deg, a.longitude_deg
        ORDER BY departures DESC
        LIMIT 200
    """).df()

    fig = px.scatter_geo(
        airport_map,
        lat="latitude",
        lon="longitude",
        size="departures",
        color="airport_type",
        hover_name="name",
        hover_data={
            "ident": True,
            "municipality": True,
            "departures": ":,",
            "latitude": False,
            "longitude": False,
        },
        color_discrete_map={
            "Barcelona": "#e4572e",
            "Other airport": "#2f6f9f",
        },
        size_max=28,
        scope="europe",
        title="European flight departures, with Barcelona highlighted",
    )

    fig.update_geos(
        showland=True,
        landcolor="#f2efe8",
        showocean=True,
        oceancolor="#dbeaf2",
        showcountries=True,
        countrycolor="#ffffff",
        lataxis_range=[34, 72],
        lonaxis_range=[-12, 35],
    )

    fig.show()
else:
    print("Airport map skipped. Set RUN_AIRPORT_MAP = True to display it.")

### Optional: DuckDB’s spatial extension
Extensions add capabilities to DuckDB’s SQL engine.
<br>`INSTALL spatial` makes the extension available;
<br>`LOAD spatial` enables its functions for this connection.
Set `RUN_SPATIAL_SQL = True` to calculate distances from Barcelona using the airport
reference CSV. The result lists 20 nearby airports, including Barcelona itself at zero.
<br>These are great-circle distances between coordinates. They do not represent actual
flight routes or prove that a flight operated between the airports.

In [ ]:
RUN_SPATIAL_SQL = True

if RUN_SPATIAL_SQL:
    con.execute("INSTALL spatial")
    con.execute("LOAD spatial")

    con.sql(f"""
        WITH airports AS (
            SELECT ident,
                   name,
                   ST_Point(
                       try_cast(latitude_deg AS DOUBLE),
                       try_cast(longitude_deg AS DOUBLE)
                   ) AS location
            FROM read_csv_auto('{AIRPORTS}')
            WHERE latitude_deg IS NOT NULL
              AND longitude_deg IS NOT NULL
        ),
        barcelona AS (
            SELECT location
            FROM airports
            WHERE ident = 'LEBL'
        )
        SELECT a.ident,
               a.name,
               round(ST_Distance_Sphere(a.location, b.location) / 1000)
                   AS km_from_barcelona
        FROM airports AS a
        CROSS JOIN barcelona AS b
        ORDER BY km_from_barcelona
        LIMIT 20
    """).show()
else:
    print("Spatial SQL skipped. Set RUN_SPATIAL_SQL = True to run the distance example.")

## 8. Take the validated query into your next workflow
**Keep the SQL question and verify the results when moving to another engine.** DuckDB-specific syntax may need adapting.
<br>For repeatable Fabric runs, add the notebook to a pipeline. Next: `2. Friendly SQL`.